# Шаг 5. Нарисуй цифру мышкой — сеть угадывает на лету

Проверим сеть на **своих** цифрах. Рисуем мышкой на чёрном холсте, а справа сеть показывает,
что она «видит» (картинку 28 × 28) и насколько уверена в каждой цифре.

Важный урок шага: сеть хорошо работает только на данных, **похожих на те, на которых училась**.
В MNIST цифры аккуратно подготовлены — вписаны в квадрат 20 × 20 и отцентрованы. Если подать сети
сырой рисунок, она часто ошибается. Клавиша **p** включает и выключает такую подготовку — сравните.

> ⚠️ Ноутбук открывает **отдельное окно OpenCV** (может оказаться позади браузера).
> Пока окно открыто, ячейка выполняется (`[*]`). В Google Colab не работает: там откройте `05_draw_digit_colab.ipynb`.
> Нужна обученная сеть `mlp.pt` (ноутбук `03_train_mlp.ipynb`).

**Управление:** левая кнопка мыши — рисовать, **c** — стереть, **p** — подготовка «как в MNIST» вкл/выкл, **q** — выход.

**Структура ноутбука (алгоритм):**
1. Загружаем обученную сеть, создаём чёрный холст 420 × 420.
2. Обработчик мыши рисует белые круги там, где проходит курсор с зажатой кнопкой.
3. Функция `to_mnist` превращает холст в картинку 28 × 28 «как в MNIST»: обрезать → вписать в 20 × 20 → отцентровать.
4. Функция `predict` получает от сети 10 вероятностей.
5. Функция `panel` рисует правую панель: что видит сеть и столбики вероятностей.
6. Главный цикл: подготовить холст → предсказать → нарисовать → обработать клавиши.

## 1. Импорт, сеть и холст

- `canvas` — холст: массив 420 × 420 из нулей (чёрный), тип `uint8`. Рисуем белым (255) — как в MNIST: белая цифра на чёрном фоне.
- `drawing` — флаг «кнопка мыши зажата».
- `prepare` — флаг «включена подготовка как в MNIST».

In [2]:
import cv2
import numpy as np
import torch

from common import load_model

model = load_model()

SIZE = 420
canvas = np.zeros((SIZE, SIZE), np.uint8)
drawing = False
prepare = True

## 2. Рисование мышью

OpenCV вызывает функцию `on_mouse` при каждом событии мыши в окне и передаёт тип события и координаты `(x, y)`:
- `EVENT_LBUTTONDOWN` — кнопка нажата → начинаем рисовать;
- `EVENT_LBUTTONUP` — кнопка отпущена → перестаём;
- `EVENT_MOUSEMOVE` — мышь движется → если кнопка зажата, рисуем.

`cv2.circle(canvas, (x, y), 16, 255, -1)` — белый круг радиусом 16 пикселей; `-1` значит «залитый».
Из множества кругов вдоль пути мыши получается толстая линия.

`global drawing` нужен, чтобы функция меняла переменную снаружи, а не создавала свою локальную.

In [3]:
def on_mouse(event, x, y, flags, param):
    global drawing
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
    if drawing and event in (cv2.EVENT_LBUTTONDOWN, cv2.EVENT_MOUSEMOVE):
        cv2.circle(canvas, (x, y), 16, 255, -1)

## 3. Подготовка «как в MNIST»

Создатели MNIST готовили каждую цифру так: вписывали её в квадрат 20 × 20 с сохранением пропорций и ставили
**центром масс** (средней точкой всех белых пикселей) в центр поля 28 × 28. Повторяем то же самое:

1. Если подготовка выключена — просто уменьшаем весь холст до 28 × 28 (`INTER_AREA` усредняет пиксели — подходит для уменьшения).
2. `np.nonzero(img)` — координаты всех белых пикселей. Если их нет — холст пустой, возвращаем чёрный квадрат.
3. **Обрезать** пустые поля: вырезаем прямоугольник от самого верхнего до самого нижнего и от самого левого до самого правого белого пикселя.
4. **Вписать в 20 × 20**: коэффициент `k = 20 / max(h, w)` — большая сторона станет 20, меньшая уменьшится пропорционально
   (`max(1, ...)` — защита от нулевого размера у очень тонкой линии).
5. Положить цифру в середину чёрного поля 28 × 28.
6. **Сдвинуть центр масс в точку (14, 14)**: `cv2.moments` считает «моменты» картинки; `m10 / m00` и `m01 / m00` —
   координаты центра масс по x и y. Матрица сдвига `[[1, 0, dx], [0, 1, dy]]` и `cv2.warpAffine` сдвигают картинку на `(dx, dy)`.

In [4]:
def to_mnist(img):
    """Сделать из рисунка картинку 28×28, похожую на цифры MNIST.

    В MNIST цифра вписана в квадрат 20×20 и стоит центром масс посередине.
    Без этой подготовки сеть ошибается гораздо чаще — проверьте клавишей p.
    """
    if not prepare:
        return cv2.resize(img, (28, 28), interpolation=cv2.INTER_AREA)
    ys, xs = np.nonzero(img)
    if len(xs) == 0:
        return np.zeros((28, 28), np.uint8)
    digit = img[ys.min():ys.max() + 1, xs.min():xs.max() + 1]  # обрезать пустые поля
    h, w = digit.shape
    k = 20 / max(h, w)
    digit = cv2.resize(digit, (max(1, round(w * k)), max(1, round(h * k))), interpolation=cv2.INTER_AREA)
    out = np.zeros((28, 28), np.uint8)
    h, w = digit.shape
    out[(28 - h) // 2:(28 - h) // 2 + h, (28 - w) // 2:(28 - w) // 2 + w] = digit
    m = cv2.moments(out)  # сдвинуть центр масс в точку (14, 14)
    shift = np.float32([[1, 0, 14 - m["m10"] / m["m00"]], [0, 1, 14 - m["m01"] / m["m00"]]])
    return cv2.warpAffine(out, shift, (28, 28))

## 4. Предсказание

Подаём картинку сети в том же виде, что при обучении:
1. `torch.tensor(..., dtype=torch.float32)` — массив NumPy в тензор с дробными числами;
2. `.div(255)` — диапазон 0…255 в 0…1 (так делал `ToTensor` при обучении);
3. `.view(1, 1, 28, 28)` — форма «пачка из 1 картинки, 1 канал, 28 × 28».

`torch.no_grad()` отключает подсчёт градиентов — при предсказании они не нужны, так быстрее.
`torch.softmax` превращает 10 оценок сети в **вероятности**: все положительные и в сумме дают 1.
`[0]` — первая (единственная) картинка пачки.

In [5]:
def predict(small):
    x = torch.tensor(small, dtype=torch.float32).div(255).view(1, 1, 28, 28)
    with torch.no_grad():
        return torch.softmax(model(x), dim=1)[0].numpy()  # 10 вероятностей, в сумме 1

## 5. Правая панель

Функция `panel` рисует на тёмно-сером прямоугольнике 420 × 300:
1. **«Что видит сеть»** — картинку 28 × 28, увеличенную до 112 × 112. `INTER_NEAREST` копирует пиксели без сглаживания,
   поэтому видны отдельные «квадратики» — именно с таким разрешением работает сеть.
2. **10 столбиков** — по одному на цифру; длина столбика `probs[d] · 200` пикселей пропорциональна вероятности.
   Столбик самой вероятной цифры (`argmax`) — зелёный, остальные — голубые. Справа — вероятность в процентах.
3. Внизу — состояние подготовки `prepare: ON/OFF`.

Цвета в OpenCV задаются в порядке **(B, G, R)**: `(80, 200, 80)` — зелёный, `(0, 220, 255)` — жёлтый.

In [6]:
def panel(small, probs):
    """Правая панель: что видит сеть (28×28) и столбики уверенности."""
    p = np.full((SIZE, 300, 3), 30, np.uint8)
    p[12:124, 94:206] = cv2.cvtColor(cv2.resize(small, (112, 112), interpolation=cv2.INTER_NEAREST),
                                     cv2.COLOR_GRAY2BGR)
    cv2.putText(p, "network sees", (96, 142), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (180, 180, 180), 1)
    best = int(probs.argmax())
    for d in range(10):
        y = 168 + d * 24
        color = (80, 200, 80) if d == best else (200, 160, 90)
        cv2.putText(p, str(d), (14, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (230, 230, 230), 1)
        cv2.rectangle(p, (40, y + 2), (40 + int(probs[d] * 200), y + 18), color, -1)
        cv2.putText(p, f"{probs[d]:.0%}", (246, y + 15), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (230, 230, 230), 1)
    cv2.putText(p, f"prepare: {'ON' if prepare else 'OFF'}  [p]", (14, SIZE - 12),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 220, 255), 1)
    return p

## 6. Главный цикл

1. `cv2.namedWindow` создаёт окно, `cv2.setMouseCallback` привязывает к нему обработчик мыши.
2. В каждой итерации:
   - готовим холст функцией `to_mnist`;
   - если на холсте что-то нарисовано (`canvas.any()`), спрашиваем сеть, иначе все вероятности — нули;
   - слева показываем холст (переведённый в цветной, чтобы склеить с цветной панелью), справа — панель; `np.hstack` ставит их рядом;
   - `cv2.waitKey(20)` ждёт клавишу 20 мс и даёт окну обработать события мыши.
3. Клавиши: `q` или закрытие окна — выход, `c` — очистить холст (`canvas[:] = 0` — все пиксели в ноль), `p` — переключить подготовку.

In [7]:
cv2.namedWindow("draw a digit")
cv2.setMouseCallback("draw a digit", on_mouse)

while True:
    small = to_mnist(canvas)
    probs = predict(small) if canvas.any() else np.zeros(10)
    left = cv2.cvtColor(canvas, cv2.COLOR_GRAY2BGR)
    cv2.putText(left, "[c] clear  [q] quit", (10, SIZE - 12), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (120, 120, 120), 1)
    cv2.imshow("draw a digit", np.hstack([left, panel(small, probs)]))

    key = cv2.waitKey(20) & 0xFF
    if key == ord("q") or cv2.getWindowProperty("draw a digit", cv2.WND_PROP_VISIBLE) < 1:
        break
    if key == ord("c"):
        canvas[:] = 0
    if key == ord("p"):
        prepare = not prepare

cv2.destroyAllWindows()

## Попробуйте сами

1. Нарисуйте маленькую цифру в углу холста с выключенной подготовкой (`p`), потом включите её. Что изменилось?
2. Нарисуйте «семёрку» с перечёркиванием, как пишут в России. Узнаёт ли её сеть? Почему?
3. Нарисуйте что-то, что вообще не цифра (букву, смайлик). Что ответит сеть? Может ли она сказать «не знаю»?
4. Поменяйте толщину кисти (радиус `16`) на 5 и 30. Как это влияет на ответы?